# 家庭电力消耗多变量时间序列预测大作业

本 Notebook 实现了课程大作业的完整代码，包含以下关键流程：
1. **数据加载与预处理**：合并分钟级电力消耗数据和月度气象数据，完成天级重采样、特征计算（如周期性日期特征）和数据清洗，提取出 `train.csv` 和 `tes.csv`。
2. **数据集与标准化**：设计防止数据泄露的 `Standardizer`（在训练集拟合，应用于测试集），构建滑窗 Dataset（输入 90 天，预测未来 90 天或 365 天）。
3. **模型构建**：实现基础的 **LSTM** 和 **Transformer** 预测模型，并提出一种新颖的改进模型 **WeatherAwareMST (Weather-Aware Multi-Scale Transformer)**。
4. **训练与测试**：支持早停（Early Stopping）和学习率衰减，完成 3 种模型 x 2 种预测步长（90天/365天）x 5 个随机种子的完整训练和测试，收集 MSE / MAE 指标并计算均值与标准差。
5. **预测曲线可视化**：绘制测试集样本上各模型预测值与真实值的对比图。

In [1]:
# =========================================
# 0. 自动安装python包
# =========================================
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing = [pip_name for import_name, pip_name in required_packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are available.")

All required packages are available.


In [2]:
# =========================================
# 1. 导入库与全局常数设置
# =========================================
from pathlib import Path
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# 设置随机种子以保证结果可复现
SEEDS = [2026, 2027, 2028, 2029, 2030]
INPUT_LEN = 90  # 输入历史天数
HORIZONS = [90, 365]  # 预测步长：90天（短期）和365天（长期）

# 优化训练超参数，确保在 CPU 环境下快速收敛运行
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 15
PATIENCE = 4

# 设定运行设备
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 设置画图字体（支持中文）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

# 路径设置
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
for p in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DEVICE:", DEVICE)

C:\Users\enola\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\enola\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


ROOT: C:\doing\suda_课程\研一下\机器学习\期末报告\temp\ML_report_code
DEVICE: cpu


In [3]:
# =========================================
# 2. 数据载入与预处理逻辑
# =========================================

POWER_PATH = ROOT / "household_power_consumption.txt"
CLIMATE_PATH = ROOT / "climate_75_PARIS-MONTSOURIS_2006-2010_required_columns.csv"

print("正在载入原始电力数据...")
raw_power = pd.read_csv(POWER_PATH, sep=";", na_values="?", low_memory=False)

# 1. 时间戳解析
raw_power["datetime"] = pd.to_datetime(raw_power["Date"] + " " + raw_power["Time"], dayfirst=True, errors="coerce")
raw_power = raw_power.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

# 2. 转换数值类型，并插值处理缺失值
numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]
for col in numeric_cols:
    raw_power[col] = pd.to_numeric(raw_power[col], errors="coerce")
raw_power[numeric_cols] = raw_power[numeric_cols].interpolate(method="linear", limit_direction="both")

# 3. 提取日期，按天进行重采样
raw_power["date"] = raw_power["datetime"].dt.floor("D")
daily_power = raw_power.groupby("date").agg({
    "Global_active_power": "sum",
    "Global_reactive_power": "sum",
    "Sub_metering_1": "sum",
    "Sub_metering_2": "sum",
    "Sub_metering_3": "sum",
    "Voltage": "mean",
    "Global_intensity": "mean",
}).reset_index()

# 4. 计算第四能耗变量 Sub_metering_remainder
daily_power["Sub_metering_remainder"] = (
    daily_power["Global_active_power"] * 1000.0 / 60.0
    - daily_power["Sub_metering_1"]
    - daily_power["Sub_metering_2"]
    - daily_power["Sub_metering_3"]
)

# 5. 载入并合并气象特征
climate = pd.read_csv(CLIMATE_PATH, sep=";")
climate["AAAAMM"] = climate["AAAAMM"].astype(str)
climate["month"] = pd.to_datetime(climate["AAAAMM"] + "01", format="%Y%m%d")
climate_cols = ["RR", "NBJRR1", "NBJRR5", "NBJRR10", "NBJBROU"]
for col in climate_cols:
    climate[col] = pd.to_numeric(climate[col], errors="coerce")

# 气象数据中：RR为十分之一毫米。在此除以10转换为标准毫米单位，符合PDF要求
climate["RR"] = climate["RR"] / 10.0

monthly_climate = climate.groupby("month", as_index=False)[climate_cols].mean()
daily_power["month"] = daily_power["date"].values.astype("datetime64[M]")

daily = daily_power.merge(monthly_climate, on="month", how="left")
daily = daily.sort_values("date").reset_index(drop=True)
daily[climate_cols] = daily[climate_cols].interpolate(limit_direction="both").ffill().bfill()

# 6. 添加周期性日期特征
daily["dayofweek"] = daily["date"].dt.dayofweek
daily["dayofyear"] = daily["date"].dt.dayofyear
daily["dow_sin"] = np.sin(2 * np.pi * daily["dayofweek"] / 7.0)
daily["dow_cos"] = np.cos(2 * np.pi * daily["dayofweek"] / 7.0)
daily["doy_sin"] = np.sin(2 * np.pi * daily["dayofyear"] / 365.25)
daily["doy_cos"] = np.cos(2 * np.pi * daily["dayofyear"] / 365.25)

# 7. 划分训练集与测试集 (前900天为训练集，剩余542天为测试集)
train_df = daily.iloc[:900].copy().reset_index(drop=True)
test_df = daily.iloc[900:].copy().reset_index(drop=True)

# 8. 写入文件以符合PDF对 train.csv 和 test.csv 的要求
train_df.to_csv(ROOT / "train.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(ROOT / "test.csv", index=False, encoding="utf-8-sig")

print("预处理完成！")
print(f"train.csv 保存路径: {ROOT / 'train.csv'}, 样本数: {len(train_df)}")
print(f"test.csv 保存路径: {ROOT / 'test.csv'}, 样本数: {len(test_df)}")

正在载入原始电力数据...


预处理完成！
train.csv 保存路径: C:\doing\suda_课程\研一下\机器学习\期末报告\temp\ML_report_code\train.csv, 样本数: 900
test.csv 保存路径: C:\doing\suda_课程\研一下\机器学习\期末报告\temp\ML_report_code\test.csv, 样本数: 542


In [4]:
# =========================================
# 3. 辅助类与函数 (Standardizer 与 PowerWindowDataset)
# =========================================

def set_seed(seed: int):
    """设置随机数种子以保证实验可复现"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class Standardizer:
    """手动实现的Z-Score标准化器，用于对特征和标签进行归一化"""
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.mean_ = x.mean(axis=0, keepdims=True)
        self.std_ = x.std(axis=0, keepdims=True)
        # 避免标准差为0的情况
        self.std_[self.std_ < 1e-6] = 1.0
        return self

    def transform(self, x):
        return (np.asarray(x, dtype=np.float32) - self.mean_) / self.std_

    def inverse_transform(self, x):
        return np.asarray(x, dtype=np.float32) * self.std_ + self.mean_


class PowerWindowDataset(Dataset):
    """用于 PyTorch 训练的滑窗数据集"""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


TARGET_COL = "Global_active_power"
FEATURE_COLS = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3", "Sub_metering_remainder",
    "RR", "NBJRR1", "NBJRR5", "NBJRR10", "NBJBROU",
    "dow_sin", "dow_cos", "doy_sin", "doy_cos",
]

def make_windows(df: pd.DataFrame, input_len: int, horizon: int):
    """从天级时序 DataFrame 中提取滑窗数据。 X: [N, T, D], y: [N, H]"""
    values = df[FEATURE_COLS].to_numpy(dtype=np.float32)
    target = df[TARGET_COL].to_numpy(dtype=np.float32)
    dates = pd.to_datetime(df["date"]).to_numpy()

    X, y, meta = [], [], []
    max_start = len(df) - input_len - horizon + 1
    for start in range(max_start):
        input_end = start + input_len
        output_end = input_end + horizon
        X.append(values[start:input_end])
        y.append(target[input_end:output_end])
        meta.append({
            "input_start": str(pd.Timestamp(dates[start]).date()),
            "input_end": str(pd.Timestamp(dates[input_end - 1]).date()),
            "target_start": str(pd.Timestamp(dates[input_end]).date()),
            "target_end": str(pd.Timestamp(dates[output_end - 1]).date()),
        })
    return np.stack(X), np.stack(y), pd.DataFrame(meta)

In [5]:
class PositionalEncoding(nn.Module):
    """用于 Transformer 的位置编码"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class LSTMForecaster(nn.Module):
    """基础 LSTM 预测模型"""
    def __init__(self, num_features, horizon, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, horizon)
        )

    def forward(self, x):
        out, _ = self.lstm(x)       # out shape: [B, T, hidden_size]
        last_state = out[:, -1, :]  # 取最后一个时间步 [B, hidden_size]
        return self.head(last_state)


class TransformerForecaster(nn.Module):
    """标准 Transformer Encoder 预测模型"""
    def __init__(self, num_features, horizon, d_model=32, nhead=2, num_layers=1, dim_feedforward=64, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(num_features, d_model)
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, horizon)
        )

    def forward(self, x):
        z = self.pos(self.input_proj(x))  # [B, T, d_model]
        z = self.encoder(z)               # [B, T, d_model]
        pooled = z.mean(dim=1)            # 全局平均池化: [B, d_model]
        return self.head(pooled)          # [B, H]


class WeatherAwareMultiScaleTransformer(nn.Module):
    """
    提出的一种新颖改进模型：Weather-Aware Multi-Scale Transformer (WeatherAwareMST)
    设计理念：
    1. 多尺度一维卷积层（卷积核大小分别为 3, 7, 15），捕捉多时间尺度上的局部趋势（如日级、周级、双周级变化特征）。
    2. 气象条件步级门控调节器 (Step-wise Weather Gating Modulator)：将输入窗口每一天的气象条件进行投影并进行 Sigmoid 激活，
       以乘法门形式动态调节每个时间步的多尺度时序特征，在表征上体现“气象变化对电力负荷的调节作用”。
    3. 卷积路径引入层归一化 (CNN-Path LayerNorm) 并进行残差拼接，确保特征尺度平衡与稳定梯度流。
    4. Transformer 编码层：对经过气象调控后的全局多尺度表征提取长期时间依赖关系。
    """
    def __init__(self, num_features, horizon, climate_indices, d_model=32, nhead=2, dropout=0.3):
        super().__init__()
        self.climate_indices = climate_indices
        self.input_proj = nn.Linear(num_features, d_model)
        
        # 多尺度卷积神经网络，分别对应短期、周级、半月级局部周期趋势提取
        self.convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=k, padding=k // 2)
            for k in [3, 7, 15]
        ])
        self.fuse = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.norm_cnn = nn.LayerNorm(d_model)
        
        # 气象条件门控调节器：基于序列中每日的外部气候状态，动态调节特征表征的幅值
        self.weather_gate = nn.Sequential(
            nn.Linear(len(climate_indices), d_model),
            nn.Sigmoid()
        )
        
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, horizon)
        )

    def forward(self, x):
        # 基础时序特征投影: [B, T, d_model]
        base = self.input_proj(x)
        
        # 多尺度局部卷积特征融合
        conv_in = base.transpose(1, 2)  # [B, d_model, T]
        multi = [torch.relu(conv(conv_in)).transpose(1, 2) for conv in self.convs]
        # 对 CNN 融合的局部时序特征进行归一化并与 base 隐状态残差相加
        z = self.norm_cnn(self.fuse(torch.cat(multi, dim=-1))) + base  # [B, T, d_model]
        
        # 提取时间序列中每一天的气候特征进行步骤级门控控制: [B, T, n_weather_features]
        weather_features = x[:, :, self.climate_indices]
        gate = self.weather_gate(weather_features)  # [B, T, d_model]
        
        # 气象门控调节：动态调控每一天的多尺度融合特征
        z = z * (1.0 + gate)
        
        # Transformer 时序长程关系建模
        z = self.encoder(self.pos(z))
        pooled = z.mean(dim=1)  # [B, d_model]
        return self.head(pooled)  # [B, H]



In [6]:
# =========================================
# 5. 训练与测试的核心循环函数 (更新：字体放大2倍)
# =========================================

def run_one_epoch(model, loader, optimizer=None):
    """跑一个 epoch。optimizer 不为 None 表示训练模式，否则为评估模式"""
    is_train = optimizer is not None
    model.train(is_train)
    criterion = nn.MSELoss()
    total_loss, total_n = 0.0, 0

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, y)
        if is_train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item() * len(X)
        total_n += len(X)
    return total_loss / max(total_n, 1)


def predict(model, loader):
    """在评估集/测试集上输出模型预测值与真实值"""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            pred = model(X).cpu().numpy()
            preds.append(pred)
            trues.append(y.numpy())
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)


def train_model(model, train_loader, val_loader, test_loader, target_scaler, model_name, horizon, seed):
    """训练模型的完整流程：包含 AdamW、学习率衰减、早停（Early Stopping）"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val_loss = float("inf")
    best_weights = None
    bad_epochs = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = run_one_epoch(model, train_loader, optimizer)
        val_loss = run_one_epoch(model, val_loader, optimizer=None)
        scheduler.step(val_loss)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            break

    # 载入最优参数
    if best_weights is not None:
        model.load_state_dict(best_weights)

    # 保存模型权重
    torch.save(model.state_dict(), MODEL_DIR / f"{model_name}_h{horizon}_seed{seed}.pt")
    
    # 对测试集进行预测，并做逆标准化还原为原始电力数值（kW）
    pred_scaled, true_scaled = predict(model, test_loader)
    pred = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
    true = target_scaler.inverse_transform(true_scaled.reshape(-1, 1)).reshape(true_scaled.shape)
    
    # 计算指标
    mse = float(np.mean((pred - true) ** 2))
    mae = float(np.mean(np.abs(pred - true)))
    return {"mse": mse, "mae": mae, "pred": pred, "true": true, "history": history}


def plot_one_prediction(true, pred, meta, model_name, horizon, seed, sample_idx=0):
    """绘制单条预测曲线与 Ground Truth 的对比，保存并展示（字体大小增加2倍）"""
    row = meta.iloc[sample_idx]
    dates = pd.date_range(row["target_start"], periods=horizon, freq="D")
    
    plt.figure(figsize=(12, 6.5)) # 画布尺寸略微增大以适配大字体
    plt.plot(dates, true[sample_idx], label="Ground Truth (实际值)", color="#1f77b4", linewidth=2.5)
    plt.plot(dates, pred[sample_idx], label=f"{model_name} Prediction (预测值)", color="#ff7f0e", linestyle="--", linewidth=2.5)
    
    plt.title(f"{model_name} 预测曲线对比 | 预测步长 H={horizon} | 随机种子={seed}", fontsize=24, pad=15)
    plt.xlabel("日期 (Date)", fontsize=20, labelpad=10)
    plt.ylabel("总有功功率 (Global_active_power, kW)", fontsize=20, labelpad=10)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(loc="upper right", fontsize=16)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    
    fig_path = FIG_DIR / f"prediction_{model_name}_h{horizon}_seed{seed}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close()
    return fig_path



In [7]:
# =========================================
# 6a. 运行 H=90 短期预测实验（分开训练）
# =========================================

# 提取气候特征在特征列表中的索引值，供 WeatherAwareMST 的门控层定位
climate_indices = [FEATURE_COLS.index(c) for c in ["RR", "NBJRR1", "NBJRR5", "NBJRR10", "NBJBROU"]]
num_features = len(FEATURE_COLS)

def make_model(model_name, horizon):
    """模型实例生成工厂"""
    if model_name == "LSTM":
        return LSTMForecaster(num_features, horizon)
    elif model_name == "Transformer":
        return TransformerForecaster(num_features, horizon)
    elif model_name == "WeatherAwareMST":
        return WeatherAwareMultiScaleTransformer(num_features, horizon, climate_indices=climate_indices)
    raise ValueError(f"Unknown model: {model_name}")

train_df = pd.read_csv(ROOT / "train.csv")
test_df = pd.read_csv(ROOT / "test.csv")

experiment_rows = []
plot_paths = []
model_names = ["LSTM", "Transformer", "WeatherAwareMST"]

# 预测步长设置为 90
horizon = 90
print(f"\n==================== 开始进行预测步长 H={horizon} 实验 ====================")
# 1. 建立滑窗样本
X_train_raw, y_train_raw, train_meta = make_windows(train_df, INPUT_LEN, horizon)
X_test_raw, y_test_raw, test_meta = make_windows(test_df, INPUT_LEN, horizon)

# 2. 从训练样本中按时序切分 80% 作为模型拟合训练集，20% 作为防止过拟合的验证集
n_samples = len(X_train_raw)
n_fit = int(n_samples * 0.8)

X_fit_raw, y_fit_raw = X_train_raw[:n_fit], y_train_raw[:n_fit]
X_val_raw, y_val_raw = X_train_raw[n_fit:], y_train_raw[n_fit:]

# 3. 对特征与目标标签进行 Z-Score 标准化，严格防止测试集数据泄露
feature_scaler = Standardizer().fit(X_fit_raw.reshape(-1, num_features))
target_scaler = Standardizer().fit(y_fit_raw.reshape(-1, 1))

# 4. 进行特征的标准化转换
X_fit = feature_scaler.transform(X_fit_raw.reshape(-1, num_features)).reshape(X_fit_raw.shape)
X_val = feature_scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
X_test = feature_scaler.transform(X_test_raw.reshape(-1, num_features)).reshape(X_test_raw.shape)

# 目标变量标准化
y_fit = target_scaler.transform(y_fit_raw.reshape(-1, 1)).reshape(y_fit_raw.shape)
y_val = target_scaler.transform(y_val_raw.reshape(-1, 1)).reshape(y_val_raw.shape)
y_test = target_scaler.transform(y_test_raw.reshape(-1, 1)).reshape(y_test_raw.shape)

# 5. 构建 DataLoader 载入管道
train_loader = DataLoader(PowerWindowDataset(X_fit, y_fit), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(PowerWindowDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(PowerWindowDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

for model_name in model_names:
    for seed in SEEDS:
        set_seed(seed)
        model = make_model(model_name, horizon).to(DEVICE)
        
        start_time = time.time()
        result = train_model(model, train_loader, val_loader, test_loader, target_scaler, model_name, horizon, seed)
        elapsed = time.time() - start_time
        
        experiment_rows.append({
            "model": model_name,
            "horizon": horizon,
            "seed": seed,
            "mse": result["mse"],
            "mae": result["mae"],
            "seconds": elapsed,
        })
        print(f"  [Model: {model_name:15s} | H={horizon:3d} | Seed={seed}] MSE: {result['mse']:.4f} | MAE: {result['mae']:.4f} | 用时: {elapsed:.1f}s")
        
        # 使用第一个种子绘制预测图以展示曲线对比效果
        if seed == SEEDS[0]:
            fig_path = plot_one_prediction(result["true"], result["pred"], test_meta, model_name, horizon, seed, sample_idx=0)
            plot_paths.append(fig_path)




==================== 开始进行预测步长 H=90 实验 ====================


  [Model: LSTM            | H= 90 | Seed=2026] MSE: 165305.5625 | MAE: 314.2178 | 用时: 4.9s


  [Model: LSTM            | H= 90 | Seed=2027] MSE: 163094.4844 | MAE: 311.4585 | 用时: 2.9s


  [Model: LSTM            | H= 90 | Seed=2028] MSE: 177810.7500 | MAE: 326.6693 | 用时: 3.3s


  [Model: LSTM            | H= 90 | Seed=2029] MSE: 162991.2344 | MAE: 311.5160 | 用时: 2.5s


...

  [Model: WeatherAwareMST | H= 90 | Seed=2026] MSE: 161423.0781 | MAE: 311.6422 | 用时: 26.7s


  [Model: WeatherAwareMST | H= 90 | Seed=2027] MSE: 171491.7188 | MAE: 325.7485 | 用时: 15.0s


  [Model: WeatherAwareMST | H= 90 | Seed=2028] MSE: 162436.4844 | MAE: 309.5897 | 用时: 14.1s


  [Model: WeatherAwareMST | H= 90 | Seed=2029] MSE: 155712.2031 | MAE: 304.2429 | 用时: 20.0s


  [Model: WeatherAwareMST | H= 90 | Seed=2030] MSE: 169728.9219 | MAE: 324.5051 | 用时: 23.1s


In [8]:
# =========================================
# 6b. 运行 H=365 长期预测实验（分开训练）
# =========================================

# 预测步长设置为 365
horizon = 365
print(f"\n==================== 开始进行预测步长 H={horizon} 实验 ====================")
# 1. 建立滑窗样本
X_train_raw, y_train_raw, train_meta = make_windows(train_df, INPUT_LEN, horizon)
X_test_raw, y_test_raw, test_meta = make_windows(test_df, INPUT_LEN, horizon)

# 2. 从训练样本中按时序切分 80% 作为模型拟合训练集，20% 作为防止过拟合的验证集
n_samples = len(X_train_raw)
n_fit = int(n_samples * 0.8)

X_fit_raw, y_fit_raw = X_train_raw[:n_fit], y_train_raw[:n_fit]
X_val_raw, y_val_raw = X_train_raw[n_fit:], y_train_raw[n_fit:]

# 3. 对特征与目标标签进行 Z-Score 标准化，严格防止测试集数据泄露
feature_scaler = Standardizer().fit(X_fit_raw.reshape(-1, num_features))
target_scaler = Standardizer().fit(y_fit_raw.reshape(-1, 1))

# 4. 进行特征的标准化转换
X_fit = feature_scaler.transform(X_fit_raw.reshape(-1, num_features)).reshape(X_fit_raw.shape)
X_val = feature_scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape)
X_test = feature_scaler.transform(X_test_raw.reshape(-1, num_features)).reshape(X_test_raw.shape)

# 目标变量标准化
y_fit = target_scaler.transform(y_fit_raw.reshape(-1, 1)).reshape(y_fit_raw.shape)
y_val = target_scaler.transform(y_val_raw.reshape(-1, 1)).reshape(y_val_raw.shape)
y_test = target_scaler.transform(y_test_raw.reshape(-1, 1)).reshape(y_test_raw.shape)

# 5. 构建 DataLoader 载入管道
train_loader = DataLoader(PowerWindowDataset(X_fit, y_fit), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(PowerWindowDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(PowerWindowDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

for model_name in model_names:
    for seed in SEEDS:
        set_seed(seed)
        model = make_model(model_name, horizon).to(DEVICE)
        
        start_time = time.time()
        result = train_model(model, train_loader, val_loader, test_loader, target_scaler, model_name, horizon, seed)
        elapsed = time.time() - start_time
        
        experiment_rows.append({
            "model": model_name,
            "horizon": horizon,
            "seed": seed,
            "mse": result["mse"],
            "mae": result["mae"],
            "seconds": elapsed,
        })
        print(f"  [Model: {model_name:15s} | H={horizon:3d} | Seed={seed}] MSE: {result['mse']:.4f} | MAE: {result['mae']:.4f} | 用时: {elapsed:.1f}s")
        
        # 使用第一个种子绘制预测图以展示曲线对比效果
        if seed == SEEDS[0]:
            fig_path = plot_one_prediction(result["true"], result["pred"], test_meta, model_name, horizon, seed, sample_idx=0)
            plot_paths.append(fig_path)

# 整理汇总实验指标并保存到 CSV 中
results = pd.DataFrame(experiment_rows)
results.to_csv(OUTPUT_DIR / "results_all_runs.csv", index=False, encoding="utf-8-sig")

summary = results.groupby(["model", "horizon"]).agg(
    mse_mean=("mse", "mean"),
    mse_std=("mse", "std"),
    mae_mean=("mae", "mean"),
    mae_std=("mae", "std"),
    seconds_mean=("seconds", "mean")
).reset_index()

summary.to_csv(OUTPUT_DIR / "results_summary.csv", index=False, encoding="utf-8-sig")
print("\n实验全部完成！")
print(f"完整记录保存路径: {OUTPUT_DIR / 'results_all_runs.csv'}")
print(f"均值/标准差汇总统计保存路径: {OUTPUT_DIR / 'results_summary.csv'}")




==================== 开始进行预测步长 H=365 实验 ====================


  [Model: LSTM            | H=365 | Seed=2026] MSE: 186118.3438 | MAE: 333.7866 | 用时: 4.0s


  [Model: LSTM            | H=365 | Seed=2027] MSE: 178118.3594 | MAE: 326.6285 | 用时: 3.6s


  [Model: LSTM            | H=365 | Seed=2028] MSE: 186428.3594 | MAE: 336.3041 | 用时: 3.2s


  [Model: LSTM            | H=365 | Seed=2029] MSE: 198427.0625 | MAE: 347.6263 | 用时: 3.1s


...

  [Model: WeatherAwareMST | H=365 | Seed=2026] MSE: 169370.2969 | MAE: 318.3267 | 用时: 15.3s


  [Model: WeatherAwareMST | H=365 | Seed=2027] MSE: 187661.1875 | MAE: 336.3451 | 用时: 17.6s


  [Model: WeatherAwareMST | H=365 | Seed=2028] MSE: 175499.5469 | MAE: 323.2565 | 用时: 19.1s


  [Model: WeatherAwareMST | H=365 | Seed=2029] MSE: 169502.5312 | MAE: 320.5083 | 用时: 13.1s


  [Model: WeatherAwareMST | H=365 | Seed=2030] MSE: 174589.9219 | MAE: 322.6684 | 用时: 13.2s

实验全部完成！
完整记录保存路径: C:\doing\suda_课程\研一下\机器学习\期末报告\temp\ML_report_code\outputs\results_all_runs.csv
均值/标准差汇总统计保存路径: C:\doing\suda_课程\研一下\机器学习\期末报告\temp\ML_report_code\outputs\results_summary.csv


In [9]:
# =========================================
# 7. 展示实验均值/标准差汇总结果
# =========================================
summary = pd.read_csv(OUTPUT_DIR / "results_summary.csv")
pd.set_option('display.float_format', lambda x: '%.4f' % x)
display(summary)

print("\n导出的预测图列表：")
for p in FIG_DIR.glob("*.png"):
    print("-", p.name)

,model,horizon,mse_mean,mse_std,mae_mean,mae_std,seconds_mean
0,LSTM,90,167921.3344,6293.1308,316.4472,6.3714,3.1777
1,LSTM,365,187384.7000,7254.5951,335.9106,7.5593,3.3913
2,Transformer,90,174260.8000,7701.3947,324.7735,10.0754,16.5651
3,Transformer,365,186953.7156,8017.4857,335.2262,9.1553,13.5224
4,WeatherAwareMST,90,164158.4813,6453.5552,315.1457,9.5136,19.7547
5,WeatherAwareMST,365,175324.6969,7451.7105,324.2210,7.0509,15.6762



导出的预测图列表：
- prediction_LSTM_h365_seed2026.png
- prediction_LSTM_h90_seed2026.png
- prediction_Transformer_h365_seed2026.png
- prediction_Transformer_h90_seed2026.png
- prediction_WeatherAwareMST_h365_seed2026.png
- prediction_WeatherAwareMST_h90_seed2026.png
